# Nested derivative operators

This notebook is a small, reproducible check for nested `DC`, `PartialD`, and `FS` declarations. It uses toy U(1) and SU(3) models so the generated branches are easy to inspect.

In [1]:
from collections import Counter
from pathlib import Path
import re
import sys

ROOT = Path.cwd()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT / "src"))

from symbolica import S
from feynpy import (
    COLOR_ADJ_INDEX,
    COLOR_FUND_INDEX,
    DC,
    FS,
    Field,
    GaugeGroup,
    GaugeRepresentation,
    LORENTZ_INDEX,
    Model,
    PartialD,
)
from symbolic.spenso_structures import gauge_generator, structure_constant

mu, nu, rho, a = S("mu"), S("nu"), S("rho"), S("a")
g, q = S("g"), S("q")

In [2]:
ANSI_ESCAPE_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")


def clean(text):
    return ANSI_ESCAPE_RE.sub("", str(text))


def signature_names(model):
    return [signature.names for signature in model.lagrangian().vertex_signatures()]


def branch_counts(model):
    return Counter(len(term.fields) for term in model.lagrangian().terms)


def show(title, result):
    print("==========")
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", clean(expression))
            print()
    else:
        print(clean(result))
        print()


def source_lagrangian(model):
    terms = model.lagrangian_decl.source_terms
    return sum(terms[1:], terms[0]) if len(terms) > 1 else terms[0]


def show_model(label, model):
    print("==========")
    print(label)
    print()
    show("Lagrangian", source_lagrangian(model))
    print("==========")
    print("Summary")
    print("signatures:", signature_names(model))
    print("branch arities:", dict(branch_counts(model)))
    print()
    show("Feynman Rules", model.lagrangian().feynman_rule(include_delta=False, simplify=False))

## Matter-like nesting

`DC(DC(Phi, mu), mu)` transforms like `Phi`. The outer covariant derivative differentiates every expanded inner branch and adds the gauge branch in the scalar's U(1) representation.

In [3]:
photon = Field(
    "A",
    spin=1,
    self_conjugate=True,
    symbol=S("A"),
    indices=(LORENTZ_INDEX,),
)
scalar = Field(
    "Phi",
    spin=0,
    self_conjugate=False,
    symbol=S("Phi"),
    conjugate_symbol=S("Phibar"),
    quantum_numbers={"Q": q},
)
u1 = GaugeGroup(
    "U1",
    abelian=True,
    coupling=g,
    gauge_boson=photon,
    charge="Q",
)

nested_scalar = Model(
    gauge_groups=(u1,),
    fields=(photon, scalar),
    lagrangian_decl=scalar.bar * DC(DC(scalar, mu), mu),
)
show_model("Phi.bar * DC(DC(Phi, mu), mu)", nested_scalar)

assert set(signature_names(nested_scalar)) == {
    ("Phi.bar", "Phi"),
    ("Phi.bar", "Phi", "A"),
    ("Phi.bar", "Phi", "A", "A"),
}
assert branch_counts(nested_scalar) == {2: 1, 3: 3, 4: 1}

Phi.bar * DC(DC(Phi, mu), mu)

Lagrangian
Phi.bar * DC(DC(Phi, mu), mu)

Summary
signatures: [('Phi.bar', 'Phi'), ('Phi.bar', 'Phi', 'A'), ('Phi.bar', 'Phi', 'A', 'A')]
branch arities: {2: 1, 3: 3, 4: 1}

Feynman Rules
3 vertex signature(s)

Vertex: ('Phi.bar', 'Phi')
Rule: -1𝑖*pcomp(q2,mu1_int)^2*delta(Phi,Phi)*delta(Phibar,Phibar)

Vertex: ('Phi.bar', 'Phi', 'A')
Rule: -2𝑖*g*q*g(mink(4, mu1_int),mink(4, mu3))*pcomp(q2,mu1_int)*delta(A,A)*delta(Phi,Phi)*delta(Phibar,Phibar)-1𝑖*g*q*g(mink(4, mu1_int),mink(4, mu3))*pcomp(q3,mu1_int)*delta(A,A)*delta(Phi,Phi)*delta(Phibar,Phibar)

Vertex: ('Phi.bar', 'Phi', 'A', 'A')
Rule: -2𝑖*g^2*q^2*g(mink(4, mu3),mink(4, mu4))*delta(A,A)^2*delta(Phi,Phi)*delta(Phibar,Phibar)



## Partial derivatives of field strengths

`PartialD(FS(...), mu)` first expands the field strength and then applies the ordinary product rule. For the non-abelian cubic piece this creates one term where the derivative hits each gauge field.

In [4]:
gluon = Field(
    "G",
    spin=1,
    self_conjugate=True,
    symbol=S("G"),
    indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX),
)
su3_fund = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fund",
)
su3 = GaugeGroup(
    "SU3",
    abelian=False,
    coupling=g,
    gauge_boson=gluon,
    structure_constant=structure_constant,
    representations=(su3_fund,),
)

partial_fs = Model(
    gauge_groups=(su3,),
    fields=(gluon,),
    lagrangian_decl=gluon(nu, a) * PartialD(FS(su3, mu, nu, a), mu),
)
show_model("G(nu,a) * PartialD(FS(SU3, mu, nu, a), mu)", partial_fs)

cubic_targets = {
    tuple(action.target for action in term.derivatives)
    for term in partial_fs.lagrangian().terms
    if len(term.fields) == 3
}
print("  cubic derivative targets:", cubic_targets)

assert set(signature_names(partial_fs)) == {("G", "G"), ("G", "G", "G")}
assert branch_counts(partial_fs) == {2: 2, 3: 2}
assert cubic_targets == {(1,), (2,)}

G(nu,a) * PartialD(FS(SU3, mu, nu, a), mu)

Lagrangian
G * PartialD(FS(SU3, mu, nu, a), mu)

Summary
signatures: [('G', 'G'), ('G', 'G', 'G')]
branch arities: {2: 2, 3: 2}

Feynman Rules
2 vertex signature(s)

Vertex: ('G', 'G')
Rule: 1𝑖*(-g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1_int)^2*delta(G,G)^2-g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*pcomp(q2,mu1_int)^2*delta(G,G)^2)+1𝑖*(g(mink(4, mu1_int),mink(4, mu1))*g(mink(4, mu2),mink(4, mu2_int))*g(coad(8, a1),coad(8, a2))*pcomp(q2,mu1_int)*pcomp(q2,mu2_int)*delta(G,G)^2+g(mink(4, mu1_int),mink(4, mu2))*g(mink(4, mu1),mink(4, mu2_int))*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1_int)*pcomp(q1,mu2_int)*delta(G,G)^2)

Vertex: ('G', 'G', 'G')
Rule: 1𝑖*(-1𝑖*g*g(mink(4, mu1_int),mink(4, mu3))*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a3),coad(8, a2))*pcomp(q2,mu1_int)*delta(G,G)^3-1𝑖*g*g(mink(4, mu1_int),mink(4, mu3))*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a2),coad(8, a3),coad(8, a1))*pcomp(q1,mu1_

## Covariant derivatives of field strengths

`DC(FS(...), mu)` treats the field strength as adjoint-valued: it includes the partial derivative branches plus the non-abelian `g f^{abc} A_mu^b FS^c` branch. The same mechanism works when `DC` and `PartialD` are nested around `FS`.

In [5]:
covariant_fs = Model(
    gauge_groups=(su3,),
    fields=(gluon,),
    lagrangian_decl=gluon(nu, a) * DC(FS(su3, mu, nu, a), mu),
)
partial_covariant_fs = Model(
    gauge_groups=(su3,),
    fields=(gluon,),
    lagrangian_decl=gluon(nu, a) * PartialD(DC(FS(su3, mu, nu, a), rho), mu),
)
covariant_partial_fs = Model(
    gauge_groups=(su3,),
    fields=(gluon,),
    lagrangian_decl=gluon(nu, a) * DC(PartialD(FS(su3, mu, nu, a), rho), rho),
)
double_covariant_fs = Model(
    gauge_groups=(su3,),
    fields=(gluon,),
    lagrangian_decl=gluon(nu, a) * DC(DC(FS(su3, mu, nu, a), rho), rho),
)

for label, model in (
    ("DC(FS)", covariant_fs),
    ("PartialD(DC(FS))", partial_covariant_fs),
    ("DC(PartialD(FS))", covariant_partial_fs),
    ("DC(DC(FS))", double_covariant_fs),
):
    show_model(label, model)

assert branch_counts(covariant_fs) == {2: 2, 3: 4, 4: 1}
assert set(signature_names(partial_covariant_fs)) == {
    ("G", "G"),
    ("G", "G", "G"),
    ("G", "G", "G", "G"),
}
assert set(signature_names(covariant_partial_fs)) == set(signature_names(partial_covariant_fs))
assert set(signature_names(double_covariant_fs)) == {
    ("G", "G"),
    ("G", "G", "G"),
    ("G", "G", "G", "G"),
    ("G", "G", "G", "G", "G"),
}

DC(FS)

Lagrangian
G * DC(FS(SU3, mu, nu, a), mu)

Summary
signatures: [('G', 'G'), ('G', 'G', 'G'), ('G', 'G', 'G', 'G')]
branch arities: {2: 2, 3: 4, 4: 1}

Feynman Rules
3 vertex signature(s)

Vertex: ('G', 'G')
Rule: 1𝑖*(-g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1_int)^2*delta(G,G)^2-g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*pcomp(q2,mu1_int)^2*delta(G,G)^2)+1𝑖*(g(mink(4, mu1_int),mink(4, mu1))*g(mink(4, mu2),mink(4, mu2_int))*g(coad(8, a1),coad(8, a2))*pcomp(q2,mu1_int)*pcomp(q2,mu2_int)*delta(G,G)^2+g(mink(4, mu1_int),mink(4, mu2))*g(mink(4, mu1),mink(4, mu2_int))*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1_int)*pcomp(q1,mu2_int)*delta(G,G)^2)

Vertex: ('G', 'G', 'G')
Rule: 1𝑖*(1𝑖*g*g(mink(4, mu1_int),mink(4, mu3))*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q2,mu1_int)*delta(G,G)^3+1𝑖*g*g(mink(4, mu1_int),mink(4, mu3))*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a3),coad(8, a2),coad(8, a1))*pcomp(q1,mu1_int)*delta(G,G)